# Aula 22 — *Rapidly-exploring Random Trees* (RRT) em $SE(2)$ na Camada Logística Externa do Aqua Index

**Disciplina:** ECAA08 • Automática (UNIFEI) — Grupo 2  
**Projeto:** Aqua Index — SCADA da Máquina Pneumática de Envasamento de Água em Copos  
**Fronteira do Modelo:** Este *notebook* aplica o RRT cinemático **exclusivamente à camada didática externa de movimentação logística ($G_V$, Aula 18)** ao redor da célula da envasadora, acionada pelo alarme real **`R-04`** (`ZS-102 == 0`) do SCADA e intertravada pelo botão de emergência **`HS-004`** ($e_1$).

---

### Nota de Engenharia (Não Aplicabilidade na Máquina Pneumática Interna)

O RRT **não** é aplicado ao controle interno da envasadora no Automation Studio porque seus atuadores (`XV-101` a `XV-502`) são cilindros pneumáticos *on/off* batente-a-batente, o manipulador de tampas (`XV-301`/`XV-302`) possui apenas 4 estados geométricos discretos, a mesa giratória (`M-001`/`SE-001`) é 1D sequencial, a termosselagem (`HT-401`/`TIT-401`) é uma malha SISO 1D com dosagem volumétrica fixa (`XV-202`, $150\text{ mL}$) e o ciclo industrial exige *takt time* determinístico. A verificação interna da máquina é coberta por **FSM/Grafcet** (Etapa 4) e **Árvore de Atingibilidade em Redes de Petri (Aula 24)**.

Por outro lado, o reabastecimento do magazine `MAG-100` a partir de `ESTOQUE_COPOS` por um carrinho elétrico não-holonômico em um galpão com obstáculos e zonas de exclusão SCADA é um problema contínuo legítimo em $SE(2) = \mathbb{R}^2 \times (-\pi, \pi]$.

In [ ]:
import math
import time
import numpy as np

# Dependências opcionais para visualização e exportação em grafo NetworkX
try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    nx = None
    HAS_NX = False

try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    plt.rcParams["figure.dpi"] = 120
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.25
    HAS_MPL = True
except ImportError:
    plt = None
    patches = None
    HAS_MPL = False

# =============================================================================
# PARÂMETROS DO GALPÃO DIDÁTICO (CAMADA LOGÍSTICA EXTERNA G_V — AULA 18)
# =============================================================================
WORLD_W = 16.0          # Largura do galpão didático (m)
WORLD_H = 12.0          # Profundidade do galpão didático (m)

# Parâmetros físicos do carrinho abastecedor não-holonômico (modelo bicicleta)
L_WHEELBASE = 0.90      # Entre-eixos L (m)
R_VEICULO = 0.45        # Raio envoltório de segurança (margem dos obstáculos, m)
V_MIN = 0.35            # Velocidade linear mínima de avanço (m/s)
V_MAX = 1.20            # Velocidade linear máxima de avanço (m/s)
DELTA_MAX = 0.60        # Esterçamento máximo |delta| <= 0.60 rad (~34.4 deg)
R_MIN_CURVA = L_WHEELBASE / math.tan(DELTA_MAX)

# Obstáculos físicos da célula ao redor da envasadora Aqua Index
MESA_CENTER = np.array([8.0, 6.2], dtype=float)   # Centro da célula MESA-001 + E1..E5
MESA_RADIUS = 1.75                                 # Raio físico da célula (m)

RECT_OBSTACLES = [
    # (tag, x_min, y_min, x_max, y_max, descricao)
    ("PLC-001",     4.3, 7.4,  5.5, 8.8, "Painel Central CLP/SCADA (Setor 000)"),
    ("EST-500",     9.8, 4.5, 13.6, 5.7, "Esteira de Saída pós-E5 (Setor 500)"),
    ("PALETE_TEMP", 4.2, 2.5,  5.8, 4.3, "Palete/Coluna temporária no corredor direto"),
]

# Perímetro da zona de exclusão ao redor de MESA-001 (engloba doca do MAG-100)
ZONA_EXCLUSAO_RADIUS = 6.2

# Poses operacionais (x [m], y [m], theta [rad])
POSE_ESTOQUE_COPOS = np.array([1.6, 1.8, 0.0], dtype=float)
POSE_DOCA_MAG100 = np.array([13.2, 9.6, 0.0], dtype=float)
TOL_XY = 0.45           # Tolerância de posição no acoplamento (m)
TOL_THETA = 0.35        # Tolerância angular no acoplamento (rad ~ 20 deg)

print("Configuração da Camada Logística Externa (G_V — Aqua Index) carregada:")
print(f"  - Galpão didático       : {WORLD_W:.1f} m x {WORLD_H:.1f} m")
print(f"  - Veículo (L / R_v)     : L = {L_WHEELBASE:.2f} m, R_v = {R_VEICULO:.2f} m")
print(f"  - Limites cinemáticos   : v in [{V_MIN:.2f}, {V_MAX:.2f}] m/s, |delta| <= {math.degrees(DELTA_MAX):.1f} deg")
print(f"  - Raio mínimo de curva  : R_min = {R_MIN_CURVA:.2f} m")
print(f"  - NetworkX / Matplotlib : HAS_NX={HAS_NX}, HAS_MPL={HAS_MPL}")

### 1. Dinâmica Cinemática em $SE(2)$, Integração RK4 e Barreira Booleana $\Phi_{\text{SCADA}}$

O estado contínuo do carrinho abastecedor é $\mathbf{x} = [x, y, \theta]^T \in SE(2)$, regido pela cinemática não-holonômica de bicicleta:

$$\dot{x} = v \cos\theta, \qquad \dot{y} = v \sin\theta, \qquad \dot{\theta} = \frac{v}{L} \tan\delta$$

A cada passo de integração $\Delta t = 0{,}8\text{ s}$ (subdividido em $n_{\text{sub}} = 10$ subpassos RK4), avaliamos a barreira lógica do supervisório:

$$\Phi_{\text{SCADA}}(\mathbf{x}, e_1) = e_1 \lor \text{colisao\_fisica}(\mathbf{x}) \lor \text{zona\_exclusao}(\mathbf{x}, e_1)$$

onde $e_1$ representa o botão de parada de emergência **`HS-004`** da envasadora (Setor 000). Se $\Phi_{\text{SCADA}} = 1$ em qualquer subpasso intermediário $\tau \in [0, \Delta t]$, o ramo é imediatamente podado.

In [ ]:
def wrap_angle(angle: float) -> float:
    """Normaliza ângulo para o intervalo (-pi, pi]."""
    wrapped = (angle + math.pi) % (2.0 * math.pi) - math.pi
    return math.pi if wrapped == -math.pi else wrapped


def verificar_colisao_fisica(x: np.ndarray):
    """
    Verifica colisão geométrica com paredes do galpão e obstáculos físicos
    inflados pelo raio envoltório R_VEICULO.
    Retorna (colidiu: bool, tag_obstaculo: str | None).
    """
    px, py = float(x[0]), float(x[1])

    # 1. Limites físicos do galpão descontando o raio do veículo
    if px < R_VEICULO or px > WORLD_W - R_VEICULO or py < R_VEICULO or py > WORLD_H - R_VEICULO:
        return True, "LIMITE_GALPAO"

    # 2. Célula da Envasadora (MESA-001 + Estações E1-E5)
    if math.hypot(px - MESA_CENTER[0], py - MESA_CENTER[1]) <= (MESA_RADIUS + R_VEICULO):
        return True, "MESA-001_E1-E5"

    # 3. Obstáculos retangulares (distância ponto-retângulo <= R_VEICULO)
    for tag, x1, y1, x2, y2, _ in RECT_OBSTACLES:
        cx = min(max(px, x1), x2)
        cy = min(max(py, y1), y2)
        if math.hypot(px - cx, py - cy) <= R_VEICULO:
            return True, tag

    return False, None


def avaliar_phi_scada(x: np.ndarray, hs_004: int = 0, zona_exclusao_ativa: bool = False):
    """
    Avalia a barreira booleana SCADA:
        Phi_SCADA = e1 OR colisao_fisica OR zona_exclusao
    Retorna (phi_scada: int, proposicoes: dict, motivo_obstaculo: str | None).
    """
    col_fisica, obs_tag = verificar_colisao_fisica(x)
    dist_celula = math.hypot(float(x[0]) - MESA_CENTER[0], float(x[1]) - MESA_CENTER[1])
    em_zona_exclusao = (hs_004 == 1 or zona_exclusao_ativa) and (dist_celula <= ZONA_EXCLUSAO_RADIUS)

    props = {
        "e1": int(hs_004 == 1),
        "colisao_fisica": int(col_fisica),
        "zona_exclusao": int(em_zona_exclusao),
    }
    phi_scada = int(props["e1"] or props["colisao_fisica"] or props["zona_exclusao"])
    motivo = obs_tag or ("HS-004_EMERGENCIA" if props["e1"] else ("ZONA_EXCLUSAO_CELULA" if props["zona_exclusao"] else None))
    return phi_scada, props, motivo


def cinematica_bicicleta_se2(x: np.ndarray, v: float, delta: float) -> np.ndarray:
    """EDO cinemática não-holonômica em SE(2): [dx/dt, dy/dt, dtheta/dt]."""
    theta = float(x[2])
    return np.array([
        v * math.cos(theta),
        v * math.sin(theta),
        (v / L_WHEELBASE) * math.tan(delta),
    ], dtype=float)


def integrar_rk4_seguro(
    x0: np.ndarray,
    v: float,
    delta: float,
    dt: float = 0.8,
    n_sub: int = 10,
    hs_004: int = 0,
    zona_exclusao_ativa: bool = False,
    ignorar_barreiras: bool = False,
):
    """
    Integra a cinemática SE(2) por Runge-Kutta de 4ª Ordem (RK4) em n_sub subpassos,
    verificando Phi_SCADA == 0 continuamente em todo tau in [0, dt].
    """
    h = dt / n_sub
    x = x0.astype(float).copy()
    traj = [x.copy()]

    if not ignorar_barreiras:
        phi, _, motivo = avaliar_phi_scada(x, hs_004=hs_004, zona_exclusao_ativa=zona_exclusao_ativa)
        if phi == 1:
            return None, False, motivo

    for _ in range(n_sub):
        k1 = cinematica_bicicleta_se2(x, v, delta)
        k2 = cinematica_bicicleta_se2(x + 0.5 * h * k1, v, delta)
        k3 = cinematica_bicicleta_se2(x + 0.5 * h * k2, v, delta)
        k4 = cinematica_bicicleta_se2(x + h * k3, v, delta)
        x = x + (h / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
        x[2] = wrap_angle(float(x[2]))

        if not ignorar_barreiras:
            phi, _, motivo = avaliar_phi_scada(x, hs_004=hs_004, zona_exclusao_ativa=zona_exclusao_ativa)
            if phi == 1:
                return None, False, motivo
        traj.append(x.copy())

    return np.array(traj, dtype=float), True, None

print("Modelo cinemático SE(2) (RK4) e barreira booleana Phi_SCADA prontos.")

### 2. Métrica Ponderada em $SE(2)$, Arborescência $\mathcal{T}=(V,E)$ e Algoritmo RRT

Na busca do vizinho mais próximo (**`NEAREST`**), combinamos distância euclidiana no plano $(x, y)$ com a distância geodésica angular em $S^1$:

$$d(\mathbf{x}_a, \mathbf{x}_b) = \sqrt{(x_a - x_b)^2 + (y_a - y_b)^2} + w_\theta \cdot \left|\operatorname{wrap}(\theta_a - \theta_b)\right|$$

A estrutura `ArvoreRRT` mantém a árvore direcionada enraizada $\mathcal{T} = (V, E)$ em Python/NumPy puro (com conversão automática para `nx.DiGraph` quando `networkx` estiver instalado) e audita explicitamente as propriedades de Matemática Discreta:
- $\vert E \vert = \vert V \vert - 1$;
- Grau de entrada $\deg^-(v_0) = 0$ na raiz e $\deg^-(v) = 1$ para todo $v \in V \setminus \{v_0\}$;
- Ausência de ciclos e alcançabilidade de 100% dos vértices a partir da raiz $v_0$.

In [ ]:
def distancia_se2(xa: np.ndarray, xb: np.ndarray, w_theta: float = 0.75) -> float:
    """Métrica híbrida ponderada em SE(2): distância euclidiana + w_theta * |wrap(dtheta)|."""
    d_xy = math.hypot(float(xa[0] - xb[0]), float(xa[1] - xb[1]))
    d_ang = abs(wrap_angle(float(xa[2] - xb[2])))
    return d_xy + w_theta * d_ang


class ArvoreRRT:
    """Estrutura de arborescência direcionada enraizada T = (V, E) em SE(2)."""

    def __init__(self, x_root: np.ndarray):
        self.nodes = [{
            "id": 0,
            "x": x_root.astype(float).copy(),
            "parent": None,
            "time": 0.0,
            "depth": 0,
        }]
        self.edges = []  # lista de dicts {src, dst, u, traj, dt}
        self.children = {0: []}
        self.edge_map = {}

    def add_node(self, parent_id: int, x_new: np.ndarray, u, traj: np.ndarray, dt: float) -> int:
        new_id = len(self.nodes)
        parent = self.nodes[parent_id]
        self.nodes.append({
            "id": new_id,
            "x": x_new.astype(float).copy(),
            "parent": parent_id,
            "time": parent["time"] + dt,
            "depth": parent["depth"] + 1,
        })
        edge = {"src": parent_id, "dst": new_id, "u": u, "traj": traj, "dt": dt}
        self.edges.append(edge)
        self.children[parent_id].append(new_id)
        self.children[new_id] = []
        self.edge_map[(parent_id, new_id)] = edge
        return new_id

    def backtrack(self, goal_id: int):
        """Reconstrução única do caminho da raiz (0) até goal_id."""
        caminho = []
        atual = goal_id
        while atual is not None:
            caminho.append(atual)
            atual = self.nodes[atual]["parent"]
        caminho.reverse()
        return caminho

    def to_networkx(self):
        """Exporta para nx.DiGraph quando NetworkX estiver disponível."""
        if not HAS_NX:
            return None
        G = nx.DiGraph()
        for n in self.nodes:
            G.add_node(n["id"], x=n["x"], time=n["time"], depth=n["depth"])
        for e in self.edges:
            G.add_edge(e["src"], e["dst"], u=e["u"], traj=e["traj"], dt=e["dt"])
        return G

    def auditar_arborescencia(self):
        """
        Verifica rigorosamente as propriedades de árvore direcionada enraizada:
        |E| == |V| - 1, in_degree(raiz)=0, in_degree(v!=raiz)=1, sem ciclos.
        """
        num_v = len(self.nodes)
        num_e = len(self.edges)
        in_deg = [0] * num_v
        out_deg = [0] * num_v
        for e in self.edges:
            out_deg[e["src"]] += 1
            in_deg[e["dst"]] += 1

        # Busca em largura a partir da raiz para confirmar conectividade e aciclicidade
        visitados = set()
        fila = [0]
        while fila:
            u = fila.pop(0)
            if u in visitados:
                continue
            visitados.add(u)
            fila.extend(self.children[u])

        graus_entrada_ok = (in_deg[0] == 0) and all(d == 1 for d in in_deg[1:])
        conectada_sem_ciclos = (len(visitados) == num_v) and (num_e == num_v - 1)
        e_arborescencia = graus_entrada_ok and conectada_sem_ciclos

        if HAS_NX:
            G = self.to_networkx()
            e_arborescencia = e_arborescencia and nx.is_arborescence(G)

        return {
            "num_v": num_v,
            "num_e": num_e,
            "e_igual_v_menos_1": (num_e == num_v - 1),
            "in_degree_raiz": in_deg[0],
            "in_degree_demais_ok": all(d == 1 for d in in_deg[1:]),
            "todos_alcancaveis": (len(visitados) == num_v),
            "is_arborescence": e_arborescencia,
            "profundidade_max": max(n["depth"] for n in self.nodes),
            "grau_saida_max": max(out_deg),
            "grau_saida_medio": float(np.mean(out_deg)),
        }


def executar_rrt_se2(
    x_init: np.ndarray,
    x_goal: np.ndarray,
    tol_xy: float = TOL_XY,
    tol_theta: float = TOL_THETA,
    k_max: int = 2500,
    p_goal: float = 0.15,
    n_candidates: int = 12,
    dt: float = 0.8,
    w_theta: float = 0.75,
    zs_102: int = 0,
    hs_004: int = 0,
    zona_exclusao_ativa: bool = False,
    seed: int = 2026,
):
    """
    Executa o planejador RRT cinemático em SE(2) para o reabastecimento logístico
    disparado por R-04 (ZS-102 == 0) e supervisionado por Phi_SCADA (HS-004).
    """
    t0 = time.perf_counter()
    rng = np.random.default_rng(seed)
    arvore = ArvoreRRT(x_init)

    # Verificação de gatilho SCADA (R-04) e intertravamento inicial (Phi_SCADA)
    alarme_r04 = (zs_102 == 0)
    if not alarme_r04:
        return {
            "sucesso": False,
            "status": "SEM_DEMANDA_R04",
            "arvore": arvore,
            "goal_id": None,
            "caminho": [],
            "ramos_podados": 0,
            "podas_por_motivo": {},
            "tempo_s": time.perf_counter() - t0,
        }

    phi0, props0, motivo0 = avaliar_phi_scada(x_init, hs_004=hs_004, zona_exclusao_ativa=zona_exclusao_ativa)
    if phi0 == 1:
        return {
            "sucesso": False,
            "status": "ABORTO_EMERGENCIA" if props0["e1"] == 1 else "ORIGEM_BLOQUEADA",
            "arvore": arvore,
            "goal_id": None,
            "caminho": [],
            "ramos_podados": 1,
            "podas_por_motivo": {motivo0: 1},
            "tempo_s": time.perf_counter() - t0,
        }

    goal_id = None
    ramos_podados = 0
    podas_por_motivo = {}

    for _ in range(1, k_max + 1):
        # 1. SAMPLE com Goal Bias
        if rng.random() < p_goal:
            x_rand = x_goal.copy()
        else:
            x_rand = np.array([
                rng.uniform(0.5, WORLD_W - 0.5),
                rng.uniform(0.5, WORLD_H - 0.5),
                rng.uniform(-math.pi, math.pi),
            ], dtype=float)

        # 2. NEAREST sob métrica híbrida SE(2)
        best_idx = min(
            range(len(arvore.nodes)),
            key=lambda i: distancia_se2(arvore.nodes[i]["x"], x_rand, w_theta=w_theta),
        )
        x_near = arvore.nodes[best_idx]["x"]

        # 3. STEER cinemático (RK4) + 4. COLLISION_FREE & Phi_SCADA
        best_traj = None
        best_u = None
        best_dist = float("inf")

        for _ in range(n_candidates):
            v_cand = float(rng.uniform(V_MIN, V_MAX))
            delta_cand = float(rng.uniform(-DELTA_MAX, DELTA_MAX))
            traj, safe, motivo = integrar_rk4_seguro(
                x_near, v_cand, delta_cand, dt=dt, n_sub=10,
                hs_004=hs_004, zona_exclusao_ativa=zona_exclusao_ativa
            )
            if not safe:
                ramos_podados += 1
                podas_por_motivo[motivo] = podas_por_motivo.get(motivo, 0) + 1
                continue

            d_new = distancia_se2(traj[-1], x_rand, w_theta=w_theta)
            if d_new < best_dist:
                best_dist = d_new
                best_traj = traj
                best_u = (v_cand, delta_cand)

        if best_traj is not None:
            new_id = arvore.add_node(best_idx, best_traj[-1], best_u, best_traj, dt)
            x_new = best_traj[-1]
            err_xy = math.hypot(float(x_new[0] - x_goal[0]), float(x_new[1] - x_goal[1]))
            err_th = abs(wrap_angle(float(x_new[2] - x_goal[2])))
            if err_xy <= tol_xy and err_th <= tol_theta:
                goal_id = new_id
                break

    caminho = arvore.backtrack(goal_id) if goal_id is not None else []
    if goal_id is not None:
        status = "SUCESSO"
    elif zona_exclusao_ativa:
        status = "BLOQUEADO_ZONA_EXCLUSAO"
    else:
        status = "TIMEOUT"

    return {
        "sucesso": goal_id is not None,
        "status": status,
        "arvore": arvore,
        "goal_id": goal_id,
        "caminho": caminho,
        "ramos_podados": ramos_podados,
        "podas_por_motivo": podas_por_motivo,
        "tempo_s": time.perf_counter() - t0,
    }


def avaliar_manobra_linha_reta(x_init: np.ndarray, x_goal: np.ndarray, n_pontos: int = 120):
    """Avalia a manobra direta (linha reta) entre ESTOQUE_COPOS e MAG-100."""
    pontos = np.linspace(x_init[:2], x_goal[:2], n_pontos)
    obstaculos_atingidos = []
    primeiro_impacto = None
    for pt in pontos:
        col, obs = verificar_colisao_fisica(np.array([pt[0], pt[1], 0.0]))
        if col:
            if primeiro_impacto is None:
                primeiro_impacto = (float(pt[0]), float(pt[1]), obs)
            if obs not in obstaculos_atingidos:
                obstaculos_atingidos.append(obs)
    return {
        "pontos": pontos,
        "colidiu": len(obstaculos_atingidos) > 0,
        "obstaculos": obstaculos_atingidos,
        "primeiro_impacto": primeiro_impacto,
    }

print("Núcleo RRT SE(2) e auditor de arborescência definidos.")

### 3. Cenário 1 — Reabastecimento Nominal (`ZS-102 == 0` / Alarme `R-04`, `HS-004 == 0`)

O sensor capacitivo **`ZS-102`** na Estação 1 sinaliza falta de copo (`R-04`), enquanto o botão de emergência **`HS-004`** permanece desacionado ($e_1 = 0$). O planejador deve levar o veículo de **`ESTOQUE_COPOS`** $(1{,}60;\; 1{,}80;\; 0{,}0^\circ)$ até a doca **`MAG-100`** $(13{,}20;\; 9{,}60;\; 0{,}0^\circ)$, comparando o plano RRT contra a manobra direta em linha reta.

In [ ]:
# 1. Execução do RRT SE(2) no Cenário 1 (nominal)
res1 = executar_rrt_se2(
    x_init=POSE_ESTOQUE_COPOS,
    x_goal=POSE_DOCA_MAG100,
    zs_102=0,      # Alarme R-04 ativo (MAG-100 vazio)
    hs_004=0,      # Emergência desacionada (e1 = 0)
    seed=2026,
)
arvore1 = res1["arvore"]
audit1 = arvore1.auditar_arborescencia()
reta1 = avaliar_manobra_linha_reta(POSE_ESTOQUE_COPOS, POSE_DOCA_MAG100)

# 2. Verificação rigorosa de todas as propriedades matemáticas e de segurança (asserts)
assert res1["sucesso"] is True, "Cenário 1 deve convergir até a doca MAG-100."
assert res1["status"] == "SUCESSO"
assert audit1["e_igual_v_menos_1"] is True, "Árvore deve satisfazer |E| == |V| - 1."
assert audit1["in_degree_raiz"] == 0, "Raiz deve ter grau de entrada 0."
assert audit1["in_degree_demais_ok"] is True, "Demais vértices devem ter grau de entrada 1."
assert audit1["is_arborescence"] is True, "T = (V, E) deve ser uma arborescência enraizada."
assert reta1["colidiu"] is True, "Manobra em linha reta deve colidir com obstáculos do galpão."
assert "PALETE_TEMP" in reta1["obstaculos"] and "MESA-001_E1-E5" in reta1["obstaculos"]

# Auditar cada subpasso contínuo RK4 da trajetória reconstruída
caminho1 = res1["caminho"]
comprimento_total_m = 0.0
for i in range(len(caminho1) - 1):
    edge = arvore1.edge_map[(caminho1[i], caminho1[i + 1])]
    traj_seg = edge["traj"]
    for pt in traj_seg:
        phi, _, _ = avaliar_phi_scada(pt, hs_004=0)
        assert phi == 0, "Violação de Phi_SCADA encontrada em subpasso da trajetória RRT!"
    diffs = np.diff(traj_seg[:, :2], axis=0)
    comprimento_total_m += float(np.sum(np.hypot(diffs[:, 0], diffs[:, 1])))

x_final1 = arvore1.nodes[res1["goal_id"]]["x"]
erro_xy1 = math.hypot(float(x_final1[0] - POSE_DOCA_MAG100[0]), float(x_final1[1] - POSE_DOCA_MAG100[1]))
erro_th1 = abs(wrap_angle(float(x_final1[2] - POSE_DOCA_MAG100[2])))
assert erro_xy1 <= TOL_XY and erro_th1 <= TOL_THETA

print("=" * 78)
print("CENÁRIO 1 — REABASTECIMENTO NOMINAL (R-04 ATIVO | HS-004 = 0)")
print("=" * 78)
print(f"Status de Convergência     : {res1['status']} (Trajetória Segura Encontrada)")
print(f"Tempo de Computação        : {res1['tempo_s']:.3f} s")
print(f"Cardinalidade |V| e |E|    : |V| = {audit1['num_v']}, |E| = {audit1['num_e']} (|E| == |V| - 1: {audit1['e_igual_v_menos_1']})")
print(f"É Arborescência Enraizada? : {audit1['is_arborescence']} (in_deg raiz={audit1['in_degree_raiz']}, demais=1: {audit1['in_degree_demais_ok']})")
print(f"Profundidade Máxima        : {audit1['profundidade_max']} passos")
print(f"Grau de Saída Máximo/Médio : {audit1['grau_saida_max']} / {audit1['grau_saida_medio']:.2f}")
print(f"Ramos Podados (Phi_SCADA=1): {res1['ramos_podados']} ({res1['podas_por_motivo']})")
print(f"Rota RRT Reconstruída      : {len(caminho1)-1} passos | Tempo = {arvore1.nodes[res1['goal_id']]['time']:.1f} s | Distância = {comprimento_total_m:.2f} m")
print(f"Erro Final em MAG-100      : d_xy = {erro_xy1:.2f} m (<= {TOL_XY:.2f} m), |d_theta| = {math.degrees(erro_th1):.1f} deg (<= {math.degrees(TOL_THETA):.1f} deg)")
print(f"Baseline Linha Reta        : COLISÃO CONFIRMADA em {reta1['obstaculos']} (1º impacto em x={reta1['primeiro_impacto'][0]:.2f} m, y={reta1['primeiro_impacto'][1]:.2f} m)")
print("-" * 78)
print(f"{'Passo':>5} | {'Nó':>4} | {'t(s)':>5} | {'x(m)':>6} | {'y(m)':>6} | {'theta(°)':>8} | {'v(m/s)':>6} | {'delta(°)':>8}")
print("-" * 78)
for passo, vid in enumerate(caminho1):
    nd = arvore1.nodes[vid]
    xv = nd["x"]
    if passo < len(caminho1) - 1:
        v_cmd, d_cmd = arvore1.edge_map[(vid, caminho1[passo + 1])]["u"]
        u_str = f"{v_cmd:6.2f} | {math.degrees(d_cmd):8.1f}"
    else:
        u_str = "   --- |      ---"
    print(f"{passo:5d} | {vid:4d} | {nd['time']:5.1f} | {xv[0]:6.2f} | {xv[1]:6.2f} | {math.degrees(xv[2]):8.1f} | {u_str}")

### 4. Cenário 2 — Intertravamento SCADA Ativo (`HS-004 == 1`) e Zona de Exclusão da Célula

Validamos o comportamento *fail-safe* da barreira booleana $\Phi_{\text{SCADA}} = e_1 \lor \text{colisao\_fisica} \lor \text{zona\_exclusao}$ em dois testes complementares:
- **Cenário 2A (`HS-004 == 1`, $e_1 = 1$):** emergência geral acionada na envasadora bloqueia imediatamente o plano na origem (`ABORTO_EMERGENCIA`), sem emitir comandos ao veículo.
- **Cenário 2B (Zona de Exclusão Perimetral da Célula ativa, `zona_exclusao_ativa=True`):** mesmo que o veículo possa se mover longe da envasadora, qualquer ramo que tente invadir o perímetro de segurança em torno de `MESA-001` / `MAG-100` é podado por $\Phi_{\text{SCADA}} = 1$, impedindo o acoplamento.

In [ ]:
# Cenário 2A: Botão de Emergência HS-004 acionado (e1 = 1)
res2a = executar_rrt_se2(
    x_init=POSE_ESTOQUE_COPOS,
    x_goal=POSE_DOCA_MAG100,
    zs_102=0,
    hs_004=1,      # Emergência ATIVA -> Phi_SCADA == 1
    seed=2026,
)
audit2a = res2a["arvore"].auditar_arborescencia()

assert res2a["sucesso"] is False, "HS-004 == 1 deve bloquear imediatamente o plano!"
assert res2a["status"] == "ABORTO_EMERGENCIA"
assert len(res2a["caminho"]) == 0
assert audit2a["e_igual_v_menos_1"] is True and audit2a["num_v"] == 1

# Cenário 2B: Zona de Exclusão da Célula ativa (intervenção/bloqueio na célula)
res2b = executar_rrt_se2(
    x_init=POSE_ESTOQUE_COPOS,
    x_goal=POSE_DOCA_MAG100,
    k_max=450,
    zs_102=0,
    hs_004=0,
    zona_exclusao_ativa=True,
    seed=2026,
)
audit2b = res2b["arvore"].auditar_arborescencia()

assert res2b["sucesso"] is False, "Zona de exclusão ativa deve impedir acoplamento em MAG-100!"
assert res2b["status"] == "BLOQUEADO_ZONA_EXCLUSAO"
assert res2b["podas_por_motivo"].get("ZONA_EXCLUSAO_CELULA", 0) > 0
assert audit2b["is_arborescence"] is True and audit2b["e_igual_v_menos_1"] is True

# Confirma que nenhum vértice da árvore 2B violou a zona de exclusão
for nd in res2b["arvore"].nodes:
    dist_c = math.hypot(float(nd["x"][0] - MESA_CENTER[0]), float(nd["x"][1] - MESA_CENTER[1]))
    assert dist_c > ZONA_EXCLUSAO_RADIUS, "Nenhum nó pode invadir a zona de exclusão!"

print("=" * 78)
print("CENÁRIO 2 — INTERTRAVAMENTO SCADA (HS-004 = 1) E ZONA DE EXCLUSÃO")
print("=" * 78)
print(f"[2A] HS-004 == 1 (e1 = 1)     : sucesso={res2a['sucesso']} | status={res2a['status']} | |V|={audit2a['num_v']} | |E|={audit2a['num_e']} | podas={res2a['podas_por_motivo']}")
print(f"[2B] Zona Exclusão Perimetral : sucesso={res2b['sucesso']} | status={res2b['status']} | |V|={audit2b['num_v']} | |E|={audit2b['num_e']} | podas_zona={res2b['podas_por_motivo'].get('ZONA_EXCLUSAO_CELULA', 0)}")
print("[OK SCADA] Todos os intertravamentos de segurança foram verificados com assert.")

### 5. Visualização da Árvore RRT no Galpão Logístico ($G_V$) e Perfis de Controle

Quando `matplotlib` estiver instalado (Jupyter/Colab), a célula abaixo plota:
1. **Planta baixa do galpão logístico ($16\text{ m} \times 12\text{ m}$):** obstáculos físicos + margens $R_v = 0{,}45\text{ m}$, árvore exploratória $\mathcal{T}=(V,E)$, trajetória segura RRT (`ESTOQUE_COPOS` $\rightarrow$ `MAG-100`), linha reta colidindo e perímetro da zona de exclusão `HS-004`;
2. **Perfis temporais** da orientação $\theta(t)$, velocidade $v(t)$ e esterçamento $\delta(t)$ dentro dos limites físicos $\mathcal{U}_{\text{adm}}$.

Em ambientes sem `matplotlib` (Python 3.13 puro com NumPy), um mapa esquemático ASCII do galpão e o resumo dos limites cinemáticos são impressos automaticamente sem erro.

In [ ]:
# Série temporal da trajetória segura (Cenário 1)
t_hist, th_hist, v_hist, d_hist = [], [], [], []
for i in range(len(caminho1) - 1):
    edge = arvore1.edge_map[(caminho1[i], caminho1[i + 1])]
    t_ini = arvore1.nodes[caminho1[i]]["time"]
    traj_seg = edge["traj"]
    ts = np.linspace(t_ini, t_ini + edge["dt"], len(traj_seg))
    v_cmd, d_cmd = edge["u"]
    start = 1 if i > 0 else 0
    t_hist.extend(ts[start:])
    th_hist.extend(np.degrees(traj_seg[start:, 2]))
    v_hist.extend([v_cmd] * (len(traj_seg) - start))
    d_hist.extend([math.degrees(d_cmd)] * (len(traj_seg) - start))

if HAS_MPL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6.2), gridspec_kw={"width_ratios": [1.45, 1.0]})

    # --- SUBPLOT 1: Planta Baixa do Galpão (Camada Logística G_V) ---
    # Zona de exclusão HS-004 (tracejada)
    circ_excl = patches.Circle(
        MESA_CENTER, ZONA_EXCLUSAO_RADIUS, edgecolor="#d62728", facecolor="#d62728",
        alpha=0.06, linestyle="--", linewidth=1.4, label="Zona Exclusão (HS-004 = 1)"
    )
    ax1.add_patch(circ_excl)

    # Célula da Envasadora (MESA-001 + E1-E5)
    ax1.add_patch(patches.Circle(
        MESA_CENTER, MESA_RADIUS + R_VEICULO, edgecolor="#c0392b", facecolor="#e74c3c",
        alpha=0.14, linestyle=":", linewidth=1.0
    ))
    ax1.add_patch(patches.Circle(
        MESA_CENTER, MESA_RADIUS, edgecolor="#7f1d1d", facecolor="#d9534f",
        alpha=0.65, label="Célula Envasadora (MESA-001 + E1–E5)"
    ))
    ax1.text(MESA_CENTER[0], MESA_CENTER[1], "MESA-001\nE1–E5", color="white",
             ha="center", va="center", fontsize=8.5, fontweight="bold")

    # Obstáculos retangulares
    cores_rect = {"PLC-001": "#34495e", "EST-500": "#7f8c8d", "PALETE_TEMP": "#b7791f"}
    for tag, x1, y1, x2, y2, _ in RECT_OBSTACLES:
        ax1.add_patch(patches.Rectangle(
            (x1 - R_VEICULO, y1 - R_VEICULO), (x2 - x1) + 2 * R_VEICULO, (y2 - y1) + 2 * R_VEICULO,
            edgecolor=cores_rect[tag], facecolor=cores_rect[tag], alpha=0.12, linestyle=":"
        ))
        ax1.add_patch(patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1, edgecolor="black", facecolor=cores_rect[tag], alpha=0.72
        ))
        ax1.text((x1 + x2) / 2, (y1 + y2) / 2, tag, color="white",
                 ha="center", va="center", fontsize=7.5, fontweight="bold")

    # Árvore exploratória RRT
    for e in arvore1.edges:
        tr = e["traj"]
        ax1.plot(tr[:, 0], tr[:, 1], color="#4a90e2", alpha=0.35, linewidth=1.0)

    # Linha reta ingênua (colide)
    ax1.plot(reta1["pontos"][:, 0], reta1["pontos"][:, 1], color="#d62728",
             linestyle="-.", linewidth=2.0, label="Linha Reta (Colide)")

    # Trajetória segura RRT
    for i in range(len(caminho1) - 1):
        tr = arvore1.edge_map[(caminho1[i], caminho1[i + 1])]["traj"]
        lbl = "Trajetória Segura RRT SE(2)" if i == 0 else None
        ax1.plot(tr[:, 0], tr[:, 1], color="#ff7f0e", linewidth=3.0, zorder=5, label=lbl)

    # Origem e Doca Alvo
    ax1.scatter(POSE_ESTOQUE_COPOS[0], POSE_ESTOQUE_COPOS[1], color="#1f77b4", edgecolor="black",
                s=110, zorder=6, label="Origem: ESTOQUE_COPOS")
    ax1.add_patch(patches.Circle(
        POSE_DOCA_MAG100[:2], TOL_XY, edgecolor="#2ca02c", facecolor="#2ca02c",
        alpha=0.25, linewidth=1.5, label="Doca Alvo: MAG-100"
    ))
    ax1.scatter(x_final1[0], x_final1[1], color="#2ca02c", edgecolor="black",
                marker="*", s=140, zorder=7)

    ax1.set_title("RRT SE(2) na Camada Logística Externa (G_V) — Gatilho R-04", fontweight="bold")
    ax1.set_xlabel("x (m)")
    ax1.set_ylabel("y (m)")
    ax1.set_xlim(0, WORLD_W)
    ax1.set_ylim(0, WORLD_H)
    ax1.set_aspect("equal")
    ax1.legend(loc="lower right", fontsize=7.8, framealpha=0.92)

    # --- SUBPLOT 2: Perfis de Controle Cinemático ---
    ax2.plot(t_hist, v_hist, color="#1f77b4", linewidth=2.2, label="Velocidade v(t) [m/s]")
    ax2.axhline(V_MAX, color="#1f77b4", linestyle="--", alpha=0.5)
    ax2.axhline(V_MIN, color="#1f77b4", linestyle=":", alpha=0.5)
    ax2.set_xlabel("Tempo (s)")
    ax2.set_ylabel("Velocidade Linear v (m/s)", color="#1f77b4")
    ax2.set_title("Entradas de Controle Admissíveis (v, delta)", fontweight="bold")

    ax2_r = ax2.twinx()
    ax2_r.plot(t_hist, d_hist, color="#ff7f0e", linewidth=2.0, linestyle="-", label="Esterçamento delta(t) [°]")
    ax2_r.axhline(math.degrees(DELTA_MAX), color="#ff7f0e", linestyle="--", alpha=0.5)
    ax2_r.axhline(-math.degrees(DELTA_MAX), color="#ff7f0e", linestyle="--", alpha=0.5)
    ax2_r.set_ylabel("Ângulo de Esterçamento delta (°)", color="#ff7f0e")

    plt.tight_layout()
    plt.show()
else:
    # Fallback textual/ASCII limpo quando matplotlib não está instalado
    print("Matplotlib indisponível neste ambiente — exibindo planta esquemática ASCII (16 m x 12 m):")
    cols, rows = 48, 24
    grid = [["." for _ in range(cols)] for _ in range(rows)]

    def to_rc(px, py):
        c = min(cols - 1, max(0, int(px / WORLD_W * cols)))
        r = min(rows - 1, max(0, rows - 1 - int(py / WORLD_H * rows)))
        return r, c

    # Marca obstáculos
    for r in range(rows):
        for c in range(cols):
            px = (c + 0.5) * WORLD_W / cols
            py = (rows - 1 - r + 0.5) * WORLD_H / rows
            if math.hypot(px - MESA_CENTER[0], py - MESA_CENTER[1]) <= MESA_RADIUS:
                grid[r][c] = "M"
            else:
                for tag, x1, y1, x2, y2, _ in RECT_OBSTACLES:
                    if x1 <= px <= x2 and y1 <= py <= y2:
                        grid[r][c] = tag[0]  # P (PLC/PALETE), E (EST-500)

    # Marca trajetória segura RRT
    for i in range(len(caminho1) - 1):
        for pt in arvore1.edge_map[(caminho1[i], caminho1[i + 1])]["traj"]:
            r, c = to_rc(pt[0], pt[1])
            grid[r][c] = "*"

    r0, c0 = to_rc(POSE_ESTOQUE_COPOS[0], POSE_ESTOQUE_COPOS[1])
    rg, cg = to_rc(POSE_DOCA_MAG100[0], POSE_DOCA_MAG100[1])
    grid[r0][c0] = "S"
    grid[rg][cg] = "G"

    print("+" + "-" * cols + "+")
    for row in grid:
        print("|" + "".join(row) + "|")
    print("+" + "-" * cols + "+")
    print("Legenda: S=ESTOQUE_COPOS | G=MAG-100 | *=Trajetória RRT | M=MESA-001 | P=PLC-001/PALETE | E=EST-500")
    print(f"Limites observados na rota RRT: v in [{min(v_hist):.2f}, {max(v_hist):.2f}] m/s | delta in [{min(d_hist):.1f}°, {max(d_hist):.1f}°]")

### 6. Síntese Técnica e Rastreabilidade

1. **Honestidade de Escopo:** O controle interno da envasadora pneumática (cilindros batente-a-batente `XV-101` a `XV-502`, mesa indexada 1D e selagem térmica SISO) é determinístico e discreto — portanto **não** usa RRT, ficando a cargo de FSM/Grafcet (Etapa 4) e Redes de Petri (Aula 24).
2. **Aplicação Legítima em $SE(2)$:** Na camada logística externa ($G_V$, Aula 18), o alarme real **`R-04`** (`ZS-102 == 0`, Aulas 08, 10, 20 e 21) aciona um planejador RRT cinemático que respeita a dinâmica não-holonômica do carrinho ($R_{\min} \approx 1{,}32\text{ m}$), desvia de obstáculos físicos inflados por $R_v = 0{,}45\text{ m}$ e converge para a pose de acoplamento do `MAG-100`.
3. **Segurança Funcional via SCADA:** A barreira booleana $\Phi_{\text{SCADA}} = e_1 \lor \text{colisao\_fisica} \lor \text{zona\_exclusao}$ garante que o acionamento de **`HS-004`** ($e_1 = 1$) ou da zona de exclusão da célula aborte/bloqueie imediatamente a aproximação à envasadora.